# Chain

**Chain**(체인)은 여러 컴포넌트(요소)를 정해진 순서대로 연결하여 **복잡한 AI 작업을 단계별로 자동화**할 수 있도록 돕는 구조이다.

- 각 컴포넌트는 **이전 처리결과를 입력으로 받아 처리한 후 다음 단계로 결과를 전달**한다.
- 복잡한 작업을 여러 개의 단순한 단계로 나누고, 각 단계를 순차적으로 실행함으로써 전체 작업을 체계적으로 구성할 수 있다.

## 기본 개념

- 체인은 하나의 LLM 호출에 그치지 않고 **여러 LLM 호출이나 도구 실행등을 순차적으로 연결**하여 실행 할 수 있다.
- 예를 들어, 사용자의 질문 → 검색 → 요약 → 응답 생성 같은 일련의 작업을 체인으로 구성할 수 있다.
- 이러한 체인구조를 사용하면 **작업흐름이 명확**해지고 **코드의 재사용성**이 높아지며 **유지 보수 및 확장성이** 향상된다.

## LangChain에서의 Chain 구성 방식

LangChain은 다음 두 가지 방식을 통해 체인을 구성할 수 있다.

### 1. Off-the-shelf Chains 방식 (클래식 방식)

- LangChain에서 제공하는 **미리 정의된 Chain 클래스**(예: `LLMChain`, `SequentialChain`, `SimpleSequentialChain`)를 활용하는 방식이다.
- 각 클래스는 다양한 chain 알고리즘들을 미리 구현한 것으로 상황에 맞는 것을 선택하여 필요한 구성요소를 전달해 생생한다.
- 이 방식은 LangChain의 **초기 방식**이며, 새로운 기능 확장이나 유연한 구성에 한계가 있기 때문에 현재 **더 이상 사용되지 않음(deprecated)** 상태이다.
  - 현재 LangChain에서는 권장하지 않는 방식이다.

### 2. LCEL (LangChain Expression Language) 방식

- LCEL은 체인을 표현식(Expression) 기반의 선언적 파이프라인 방식으로 구성할 수 있도록 설계된 최신 체인 구성 방법이다. 
- 각 컴포넌트들을 `|` 연산자로 연결하여, 흐름이 자연스럽게 이어지는 형태의 체인을 구성한다.
- LCEL 방식은 간결하고 선언적인 문법을 제공하여 **직관적이고 융통성과 확장성 있는 체인 구성**이 가능하다.
- LCEL은
  - 선형적 흐름 구조를 가진다.
  - 문법이 간결하고 선언적이다.
  - 체인의 구조가 코드만 봐도 쉽게 파악된다.
  - 유연하고 확장성이 매우 뛰어나다.
- `Runnable` 기반 구조
  - LCEL방식을 구성하는 모든 컴포넌트들은 `Runnable` 이라는 공통 인터페이스를 기반으로 동작한다.
  - 체인을 구성하는 각 컴포넌트들은 `Runnable` 을 상속하여 구현하여 이를 통해 일관된 실행 인터페이스를 제공한다.
  - **공통 메소드**:
    - `invoke()`: 단일 입력에 대한 처리
    - `batch()`: 다수 입력을 묶어서 한번에 처리
    - `stream()`: 스트리밍 방식의 요청
    - `ainvoke()`, `abatch()`, `astream()`: 비동기적 처리 메소드

In [6]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
prompt = ChatPromptTemplate.from_template(
    template="{item}에 어울리는 브랜드 이름 {count}개를 만들어 주세요."
)
model = ChatOpenAI(model="gpt-5.4-nano")
parser = StrOutputParser()

In [7]:
query = prompt.invoke({"item":"가방", "count":5})
res = model.invoke(query)
result = parser.invoke(res)
print(result)

1. 어울림백（Eo-Uri Rim Bag）  
2. 다정가방（Dajeong Bag）  
3. 걸음선（Georeumseon）  
4. 노을바이브백（Noeul Vibe Bag）  
5. 가벼운하루（Gabyeoun Haru）


In [2]:
# !uv pip install langchain-classic

In [8]:
############################################################
# 기존의 Off the shell 방식 - langchain-classic 설치 필요
############################################################
from langchain_classic import LLMChain
# chain을 구성하는 요소들을 넣어서 생성.
# prompt_template-[prompt]->model-[응답]->output parser-> 최종결과
chain = LLMChain(
    prompt=prompt,
    llm=model,
    output_parser=parser
)

res = chain.invoke({"item":"가방", "count":3})
print(res)


C:\Users\Playdata\AppData\Local\Temp\ipykernel_22224\1453150728.py:7: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(


{'item': '가방', 'count': 3, 'text': '1. 루미아오라  \n2. 노빌라피오  \n3. 트래블노트'}


In [ ]:
##############################
#  LCEL
##############################
chain2 = prompt | model | parser
print(type(chain2))
res2 = chain2.invoke({"item":"TV", "count":3}) 

<class 'langchain_core.runnables.base.RunnableSequence'>


In [12]:
print(res2)

아래는 TV(텔레비전) 브랜드로 어울릴 만한 **브랜드 이름 3개**입니다.

1. **선명TV**  
2. **빛가람**  
3. **울림뷰**


In [15]:
from langchain_core.runnables import Runnable
print(isinstance(model, Runnable), isinstance(prompt, Runnable), 
      isinstance(parser, Runnable), isinstance(chain2, Runnable))

True True True True


# Runnable 타입 주요 클래스


## [Runnable](https://reference.langchain.com/python/langchain_core/runnables/#langchain_core.runnables.base.Runnable)
- LangChain의 Runnable은 실행 가능한 작업 단위를 캡슐화한 개념으로, 데이터 흐름의 각 단계를 정의하고 **체인(chain) 에 포함 되어**  복잡한 작업의 각 단계를 수행 한다.
- **Chain을 구성하는 class들**은 Runnable의 상속 받아 구현한다.
- **Prompt Template클래스**, **Chat 모델**, **Output Parser 클래스** 등 다양한 컴포넌트가 Runnable을 상속받아 구현된다.

### 주요 특징
- 작업 단위의 캡슐화:
    - Runnable은 특정 작업(예: 프롬프트 생성, LLM 호출, 출력 파싱 등)을 수행하는 독립적인 컴포넌트이다.
    - 각 컴포넌트는 독립적으로 테스트 및 재사용이 가능하며, 조합하여 복잡한 체인을 구성할 수 있다.
- 체인 연결 및 작업 흐름 관리:
    - Runnable은 체인(chain, 일련의 연결된 작업 흐름)을 구성하는 기본 단위로 사용된다.
    - LangChain Expression Language(LCEL)를 사용하면 | 연산자를 통해 여러 Runnable을 쉽게 연결할 수 있다.
    - 입력과 출력의 형식을 일관되게 유지하여 각 단계가 자연스럽게 연결된다.
- 모듈화 및 디버깅 용이성:
    - 각 단계가 명확히 분리되어 문제 발생 시 어느 단계에서 오류가 발생했는지 쉽게 확인할 수 있다.
    - 복잡한 작업을 작은 단위로 나누어 체계적으로 관리할 수 있다.
      
### Runnable의 표준 메소드
- 모든 Runnable이 구현하는 공통 메소드
    - **`invoke(input, config:RunnableConfig)->output`**: 단일 입력을 처리하여 결과를 반환.
    - **`batch(input:list, config:RunnableConfig|list[RunnableConfig]) -> list[Output]`**: 여러 입력 데이터들을 한 번에 처리.
    - **`stream(input, config:RunnableConfig) -> Iterator[Output]`**: 입력에 대해 스트리밍 방식으로 응답을 반환.
    - **`assign(**kwargs)`**:
      -  앞 Runnable의 출력 결과에 새로운 key–value 쌍의 Field 추가(assign) 하여 다음 Runnable로 전달.
      -  값으로는 Runnable 객체(LCEL체인등)나 고정 값(리터럴) 모두 가능하며, 각 항목은 실행 시 평가되어 기존 출력에 병합한다.
      -  주로 앞 단계의 출력에 부가 정보(field)를 추가하고자 할 때 사용한다. 특히 `RunnablePassthrough`와 결합해, 입력을 그대로 넘기면서 특정 field만 추가할 때 자주 사용


### Runnable의 주요 구현체(하위 클래스)

- 다음 클래스들은 기능을 제공하는 것이 아니라 **chain 구조를 다양하게 구성** 할 수 있도록 도와주는 **Runnable** 타입의 클래스들이다.

- **`RunnableSequence`**
    - 여러 `Runnable`을 순차적으로 연결하여 실행하는 구성이다.
    - 각 단계의 출력이 다음 단계의 입력으로 전달된다.
    - 보통은 LCEL 문법을 사용해서 정의한다.
      - LCEL을 사용하여 체인을 구성할 경우 자동으로 `RunnableSequence`로 변환된다.


In [18]:
from langchain_core.runnables import RunnableSequence

# chain = RunnableSequence(prompt, model, parser)
chain = prompt | model | parser
# chain
chain.invoke({"item":"물", "count":2})

'1. **물빛하우스**  \n2. **청아수림**'


- **`RunnableLambda`**
    - Lambda 표현식의 함수를 `Runnable`로 변환할 때 사용한다.    
    - 일반함수도 `RunnableLambda`로 변환할 수 있다. 단 일반 함수는 변환 없이 chain에 포함 시킬 수 있기 때문에 굳이 변환할 필요가 없다.
    - Runnable로 만들 함수 구문
        - parameter: 입력 값 1개선언.
        - return: 다음 chain에 전달할 값의 형식


In [21]:
from langchain_core.runnables import RunnableLambda

# RunnableLambda(함수) 
# 함수-파라미터(1개 -> 앞 체인으로 부터 받을 값에 맞춘다.)
#     - 리턴값 -> 다음 체인의 입력 타입에 맞게 반환.
c1 = RunnableLambda(lambda input_data:f"{input_data}에 대해서 한 문장으로 설명해줘.")
# type(c1)
c1.invoke("LLM 모델")

'LLM 모델에 대해서 한 문장으로 설명해줘.'

In [22]:
chain = c1 | model | parser
chain.invoke("LLM모델 ") #chain 호출시에는 첫번째 컴포넌트에 전달할 값을 넣어서 호출

'LLM(대규모 언어 모델)은 방대한 텍스트를 학습해 문맥을 이해하고 자연어를 생성·예측하는 인공지능 모델입니다.'

In [25]:
prompt = ChatPromptTemplate(
    messages = [
        ("system", "모든 응답은 100글자 이내로 작성해줘."),
        ("user", "{query}")
    ]
)
model = ChatOpenAI(model="gpt-5.4-mini")
parser = StrOutputParser()

# Chain구성. chain 응답: LLM 응답내용, 글자수
chain = prompt | model | parser | RunnableLambda(lambda x : (x, len(x)))

In [26]:
res = chain.invoke("AI에 대해서 설명해줘.")
print(res)

('AI는 사람처럼 학습·추론하는 컴퓨터 기술입니다. 데이터로 패턴을 배우고 예측합니다.', 47)


In [ ]:
def get_value_len(value:str):
    return value, len(value)

# 일반함수(callable)를 chain의 구성으로 포함시킬 수 있다. -> 내부적으로 Runnable로 변환되서 들어간다.
chain2 = prompt | model | parser | get_value_len   # RunnableLambda(get_value_len) 
chain2.invoke("크리스마스")

('메리 크리스마스! 🎄', 11)


-  **`RunnablePassthrough`**
    - 입력 데이터를 가공하지 않고 그대로 다음 단계로 전달하는 `Runnable`이다.
      - 앞 Runnable으로 부터 전달 받은 **입력 값을 다음 Runnable로 그대로 전달**한다.
           - `RunnablePassthrough()`
      - 입력받은 값에 **Field를 추가**해서 전달할 경우 `assign()` 메소드를 사용한다.
           - `RunnablePassthrough.assign(new_key1="new_value1", new_key2="new_value2", ..)`


In [31]:
from langchain_core.runnables import RunnablePassthrough

rp = RunnablePassthrough() # 단순히 받은 값을 다음으로 통과시킨다.

result = rp.invoke("안녕하세요")
result = rp.invoke([1, 2, 3, 4, 5])
result = rp.invoke({"a":10, "b":20})
print(result)

{'a': 10, 'b': 20}


In [32]:
# 입력받은 값(딕셔너리)에 item을 추가해서 다음으로 전달.
r1 = RunnableLambda(lambda x: "서울시 금천구 독산동")
r2 = RunnableLambda(lambda x: "010-1111-2222")

# assign(key=Runnable, ...)
## 입력받은 딕셔너리에 address와 tel_no key를 추가. value는 Runnable을 호출해서 반환된 값을 설정.
rp2 = RunnablePassthrough.assign(
    address=r1,
    tel_no=r2
)
result = rp2.invoke({"name":"홍길동"})
result

{'name': '홍길동', 'address': '서울시 금천구 독산동', 'tel_no': '010-1111-2222'}


- **`RunnableParallel`**
    - 여러 `Runnable`을 병렬로 실행한 후, 결과를 결합하여 다음 단계로 전달한다.
    - 
        ```python
        RunnableParallel(
            {
                "key1":Runnable1, 
                "key2":Runnable2,
                "key3":Runnable3, ...
            }
        )
        ```
    - 각 Runnable의 실행결과를 Value로 Dictionary를 생성해서 반환한다.
    - LCEL로 정의할 때는 Chain에 dictionary로 정의한다.



In [34]:
from langchain_core.runnables import RunnableParallel

r1 = RunnableLambda(lambda x : x + 10)
r2 = RunnableLambda(lambda x : x - 10)
r3 = RunnableLambda(lambda x : x * 10)
r4 = RunnableLambda(lambda x : x / 10)

# c = r1 | r2 | r3 | r4 
parallel = RunnableParallel(
    {
        "value1": r1,
        "value2": r2,
        "value3": r3,
        "value4": r4,
        "org_value": RunnablePassthrough() # 입력받은 값을 그대로 다음으로 넘겨야 할 경우.
    }
)

result = parallel.invoke(200)
result

{'value1': 210,
 'value2': 190,
 'value3': 2000,
 'value4': 20.0,
 'org_value': 200}

In [35]:
c = RunnablePassthrough() | {
        "value1": r1,
        "value2": r2,
        "value3": r3,
        "value4": r4,
        "org_value": RunnablePassthrough() # 입력받은 값을 그대로 다음으로 넘겨야 할 경우.
    }
c.invoke(2000)

{'value1': 2010,
 'value2': 1990,
 'value3': 20000,
 'value4': 200.0,
 'org_value': 2000}

### LCEL Chain 예제

In [ ]:
##################################################
# TODO 1
# 음식 이름을 입력하면 그 음식의 레시피를 llm이 출력하는 Chain을 LCEL 을 이용해서 구성한다.
# 입력 : 음식 이름 - recipe_chain.invoke({"food":"김치찌게"})
# 출력 : 음식의 레시피 - 김치찌게 레시피. 

# chain구성: prompt_template -> model(gpt-5-mini) -> StrOutputParser

In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

system_prompt = """
<instruction>
당신은 요리전문 AI Assistant입니다.
요청받은 음식의 레시피를 자세하고 쉽게 작성해 주세요.
출력 방법은 아래 output_format을 참고해서 응답해주세요.
</instruction>

<output_format>
- Markdown 형식으로 답변을 작성합니다.
- 응답 내용에는 다음 항목들을 포함합니다.
    - 요리 이름
    - 요리 기본정보
        - 난이도
        - 조리시간
        - 인분
    - 요리에 필요한 재료
    - 요리 방법
    - 팁
</output_format>
"""
parser = StrOutputParser()
model = ChatOpenAI(model="gpt-5.4-mini")
prompt = ChatPromptTemplate(
    messages = [
        ("system", system_prompt),
        ("user", "{food}의 레시피를 작성해 주세요.")
    ]
)

In [2]:
recipe_chain = prompt | model | parser

res = recipe_chain.invoke({"food":"김치찌게"})

In [5]:
from IPython.display import Markdown
print(res)
# Markdown(res)

# 김치찌개 레시피

## 요리 기본정보
- **난이도:** 쉬움
- **조리시간:** 약 25~30분
- **인분:** 2~3인분

## 요리에 필요한 재료
- 잘 익은 김치 1컵~1.5컵
- 돼지고기(목살 또는 앞다리살) 150g  
  - 없으면 참치 1캔 또는 두부로 대체 가능
- 두부 1/2모
- 양파 1/2개
- 대파 1대
- 다진 마늘 1큰술
- 고춧가루 1큰술
- 국간장 1큰술
- 설탕 1작은술
- 김치 국물 1/2컵
- 물 또는 육수 2.5~3컵
- 참기름 1작은술
- 식용유 1작은술
- 소금 약간
- 후추 약간

## 요리 방법
1. **재료 준비하기**  
   김치는 먹기 좋게 썰고, 돼지고기는 한입 크기로 썰어 준비합니다.  
   두부는 두툼하게 썰고, 양파는 채 썰고, 대파는 어슷썰기 합니다.

2. **고기와 김치 볶기**  
   냄비에 식용유와 참기름을 넣고 중불로 달군 뒤 돼지고기를 먼저 볶습니다.  
   고기가 어느 정도 익으면 김치를 넣고 2~3분 정도 함께 볶아 김치의 신맛과 감칠맛을 살려줍니다.

3. **양념 넣기**  
   고춧가루, 다진 마늘, 국간장, 설탕을 넣고 골고루 섞어 볶습니다.  
   김치 국물도 함께 넣어 맛을 더 진하게 만듭니다.

4. **끓이기**  
   물 또는 육수를 부은 뒤 센 불에서 끓입니다.  
   끓기 시작하면 중불로 줄여 10~15분 정도 푹 끓여줍니다.

5. **마무리 재료 넣기**  
   양파와 두부를 넣고 3~5분 더 끓입니다.  
   마지막에 대파를 넣고, 부족한 간은 소금이나 국간장으로 맞춥니다.  
   후추를 약간 뿌려 마무리합니다.

## 팁
- **김치는 잘 익은 신김치**를 사용하면 국물 맛이 훨씬 깊어집니다.
- 돼지고기는 **목살이나 앞다리살**을 사용하면 국물이 더 진하고 맛있습니다.
- 국물이 더 진한 맛을 원하면 **멸치육수나 다시마육수**를 사용해 보세요.
- 참치김치찌개로 만들 경우, 돼지고기 대신 **참치 1캔**을 마지막 단계에 넣으면 

In [6]:
res2 = recipe_chain.invoke({"food":"봉골레 파스타"})

In [7]:
print(res2)

# 봉골레 파스타 레시피

## 요리 기본정보
- **난이도:** 중
- **조리시간:** 약 20~25분
- **인분:** 2인분

## 요리에 필요한 재료
### 주재료
- 스파게티 면 160g
- 바지락 300g
- 마늘 6~8쪽
- 페페론치노 4~6개
- 올리브오일 4큰술
- 화이트와인 1/2컵(선택)
- 소금 약간
- 후추 약간
- 이탈리안 파슬리 약간

### 선택 재료
- 버터 1큰술
- 면수 1/2컵 정도
- 레몬즙 약간
- 파마산 치즈 약간(전통적인 봉골레에는 생략 가능)

## 요리 방법
1. **바지락 해감하기**
   - 바지락은 소금물에 담가 1~2시간 정도 해감한 뒤 깨끗이 씻어 주세요.
   - 껍데기끼리 비벼 씻으면 이물질 제거에 도움이 됩니다.

2. **재료 손질하기**
   - 마늘은 편으로 썰거나 얇게 다져 주세요.
   - 페페론치노는 부셔서 매운맛이 잘 우러나도록 준비합니다.
   - 파슬리는 잘게 다져 둡니다.

3. **면 삶기**
   - 끓는 물에 소금을 넣고 스파게티 면을 포장지보다 1~2분 짧게 삶아 주세요.
   - 면수는 나중에 소스 농도를 맞추는 데 사용하므로 조금 남겨 둡니다.

4. **봉골레 소스 만들기**
   - 팬에 올리브오일을 두르고 약불에서 마늘과 페페론치노를 넣어 천천히 볶아 향을 냅니다.
   - 마늘이 노릇해지기 시작하면 바지락을 넣고 중불로 올립니다.
   - 화이트와인을 넣고 뚜껑을 덮어 바지락이 입을 벌릴 때까지 익혀 주세요.

5. **면과 소스 섞기**
   - 바지락이 대부분 열리면 삶아 둔 면을 넣고 잘 섞어 줍니다.
   - 면수가 부족하면 조금씩 넣어가며 소스가 면에 잘 감기도록 볶습니다.
   - 간을 보고 소금, 후추로 맞춰 주세요.

6. **마무리**
   - 불을 끈 뒤 파슬리와 버터를 넣어 한 번 더 섞으면 풍미가 좋아집니다.
   - 취향에 따라 레몬즙을 살짝 넣어 상큼하게 마무리해도 좋습니다.

## 팁
- **바지락 해감이 중요**합니다. 해감이 덜 되면

In [ ]:
##############################################################
#  TODO 2
# 번역할 내용, 번역할 언어 를 입력하면 내용을 그 언어로 번역하는 Chain을 LCEL 을 이용해서 구성한다.
#
## 입력: 번역할 내용, 언어.  translate_chain.invoke({"content":"안녕하세요.", "language":"영어"})
## 출력: "번역할 내용"을 "언어" 로 번역한 결과 - "How are you?".

# chain구성: prompt_template -> model(gpt-5-mini) -> StrOutputParser

In [10]:
system_prompt2 = """
<instruction>
당신은 다국어가 가능한 숙련된 번역 AI Assistant입니다.
<input_data> 항목에 작성된 내용을 참고 해서 **요청된 문서**의 내용을 **요청된 언어**로 번역해 주세요.
내용의 의미를 해치지 않는 범위에서 최대한 읽기 쉽게 작성해 주세요.
</instruction>

<input_data>
- 번역할 내용: {content}
- 번역할 언어: {language}
</input_data>
"""

prompt_trans = ChatPromptTemplate.from_template(
    template=system_prompt2
)
model_trans = ChatOpenAI(model="gpt-5.4-mini")

translate_chain = prompt_trans | model_trans | StrOutputParser()


In [11]:
res = translate_chain.invoke({"content":"안녕하세요.", "language":"영어"})
print(res) 

Hello.


In [ ]:
# streamlit \ 04_multimodal \ data\ doc.pdf

In [12]:
content = """The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring significantly
less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-
to-German translation task, improving over the existing best results, including
ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task,
our model establishes a new single-model state-of-the-art BLEU score of 41.8 after
training for 3.5 days on eight GPUs, a small fraction of the training costs of the
best models from the literature. We show that the Transformer generalizes well to
other tasks by applying it successfully to English constituency parsing both with
large and limited training data."""

res = translate_chain.invoke({"content":content, "language":"한국어"})

In [13]:
print(res)

지배적인 시퀀스 변환 모델들은 인코더와 디코더를 포함하는 복잡한 순환 신경망 또는 합성곱 신경망에 기반한다. 가장 우수한 성능을 보이는 모델들은 주의(attention) 메커니즘을 통해 인코더와 디코더를 연결하기도 한다. 우리는 순환과 합성곱을 전혀 사용하지 않고, 오직 주의 메커니즘만으로 구성된 새로운 간단한 네트워크 구조인 Transformer를 제안한다. 두 개의 기계번역 과제에 대한 실험 결과, 이 모델들은 더 높은 품질을 보이면서도 더 병렬화하기 쉬우며 학습 시간도 훨씬 적게 든다는 것을 확인했다. 우리의 모델은 WMT 2014 영어-독일어 번역 과제에서 28.4 BLEU를 달성하여, 앙상블을 포함한 기존 최고 성능보다 2 BLEU 이상 향상되었다. 또한 WMT 2014 영어-프랑스어 번역 과제에서는 8개의 GPU로 3.5일 동안 학습한 후 41.8의 새로운 단일 모델 최고 BLEU 점수를 기록했는데, 이는 기존 문헌의 최고 모델들에 비해 학습 비용이 매우 작은 수준이다. 우리는 Transformer가 다른 과제에도 잘 일반화됨을 보이기 위해, 충분한 학습 데이터와 제한된 학습 데이터 모두에서 영어 구문 분석(English constituency parsing)에 성공적으로 적용했다.


In [14]:
res = translate_chain.invoke({"content":content, "language":"프랑스어"})

In [15]:
print(res)

Les modèles dominants de transduction de séquence reposent sur des réseaux neuronaux récurrents ou convolutionnels complexes, comprenant un encodeur et un décodeur. Les modèles les plus performants relient également l’encodeur et le décodeur par un mécanisme d’attention. Nous proposons une nouvelle architecture de réseau simple, le Transformer, fondée uniquement sur des mécanismes d’attention, éliminant entièrement la récurrence et les convolutions. Des expériences sur deux tâches de traduction automatique montrent que ces modèles sont de meilleure qualité, tout en étant plus parallélisables et en nécessitant beaucoup moins de temps d’entraînement. Notre modèle atteint un score BLEU de 28,4 sur la tâche de traduction anglais-allemand du WMT 2014, améliorant de plus de 2 points BLEU les meilleurs résultats existants, y compris les ensembles. Sur la tâche de traduction anglais-français du WMT 2014, notre modèle établit un nouveau record de référence pour un modèle unique avec un score BL

In [16]:
from langchain_core.runnables import Runnable

isinstance(translate_chain, Runnable)

True

### Chain과 Chain간의 연결

In [ ]:
translate_chain.invoke({"content":"요리레시피", "language":"번역할 언어이름"})

In [22]:
from operator import itemgetter

ig = itemgetter("language") # 자료구조에서 값을 조회할 index/key를 넣고 객체를 생성
a = {"language":"영어", "name":"홍길동"}
ig(a) # 자료구조를 넣어주면 생성할 때 지정한 index/key의 값을 조회해서 반환.

ig2 = itemgetter(2)
b = [10, 20, 30, 40, 50, 60]
b = [1, 2, 3 , 4, 5, 5]
ig2(b)

3

In [ ]:
# 음식 레시피를 원하는 언어로 출력하는 AI Agent가 필요.
## recipe_chain 과 translate_chain을 연결
from langchain_core.runnables import RunnableLambda
from operator import itemgetter
# {}: RunnableParallel
chain = {
    "content":recipe_chain,
    "language":itemgetter("language")
    # "language":RunnableLambda(lambda x : x['language'])
} | translate_chain

In [24]:
res = chain.invoke({"food":"김치찌게", "language":"독일어"})

In [25]:
print(res)

# Kimchi-Jjigae-Rezept

## Grundinformationen zum Gericht
- **Schwierigkeitsgrad**: Einfach
- **Zubereitungszeit**: ca. 30 Minuten
- **Portionen**: 2–3 Portionen

## Zutaten
- 1 bis 1,5 Tassen saurer Kimchi
- 150 g Schweinefleisch (Vorderkeule oder Bauchfleisch)
- 1/2 Block Tofu
- 1/2 Zwiebel
- 1 Frühlingszwiebel
- 1 bis 2 Cheongyang-Chilis
- 1 EL gehackter Knoblauch
- 1 EL Gochugaru (koreanisches Chilipulver)
- 1 TL Zucker
- 1/2 Tasse Kimchi-Sud
- 2 bis 3 Tassen Wasser
- 1 EL Speiseöl
- etwas Salz oder Suppen-Sojasauce
- etwas Pfeffer

## Zubereitung

1. **Zutaten vorbereiten**
   - Den sauren Kimchi in mundgerechte Stücke schneiden und den Tofu in dicke Scheiben schneiden.
   - Die Zwiebel in Streifen schneiden, die Frühlingszwiebel und die Cheongyang-Chilis schräg schneiden.

2. **Schweinefleisch anbraten**
   - Das Speiseöl in einen Topf geben und das Schweinefleisch bei mittlerer Hitze anbraten.
   - Sobald die Außenseite des Fleisches gar ist, den gehackten Knoblauch hinzufügen u

In [26]:
res = chain.invoke({"food":"김치찌게", "language":"베트남어"})

In [27]:
print(res)

# Công thức canh kimchi hầm

## Thông tin cơ bản
- **Độ khó:** Dễ
- **Thời gian nấu:** 30–40 phút
- **Khẩu phần:** 2–3 người

## Nguyên liệu cần chuẩn bị
- 2 cốc kimchi (kimchi đã lên men ngon)
- 150g thịt heo vai hoặc ba chỉ
- 1/2 bìa đậu phụ
- 1/2 củ hành tây
- 1 cây hành lá
- 1–2 quả ớt Cheongyang (tùy chọn)
- 1 thìa canh tỏi băm
- 1/2 cốc nước kimchi
- 2–3 cốc nước lọc hoặc nước dùng
- 1 thìa canh dầu ăn

### Gia vị
- 1 thìa canh bột ớt Hàn Quốc
- 1 thìa canh nước tương nấu canh
- Một ít muối
- Một ít tiêu

## Cách làm
1. **Sơ chế nguyên liệu**  
   Cắt kimchi thành miếng vừa ăn, đậu phụ cắt miếng vừa ăn. Hành tây thái sợi, hành lá cắt xéo. Nếu dùng ớt Cheongyang, cắt nhỏ.

2. **Xào thịt**  
   Cho dầu ăn vào nồi, thêm thịt heo vào xào trên lửa vừa. Khi thịt chín khoảng một nửa, cho tỏi băm vào xào cùng để tạo mùi thơm.

3. **Cho kimchi vào xào**  
   Thêm kimchi vào xào khoảng 2–3 phút để vị chua và vị ngọt đậm đà của kimchi được dậy lên.  
   Lúc này cho thêm bột ớt và nước tương

## 함수를 Runnable로 정의하기

### 함수 구현
- **파라미터**
   - 이전 Chain에서 출력한 값을 입력으로 받을 수 있도록 정의한다.
- **리턴값**
   - 다음 Chain으로 입력할 값을 반환하도록 구현한다.

### Runnable 타입으로 만들기
1. LCEL Chain안에 함수를 구성요소로 포함시키면, 그 함수는 자동으로 `Runnable` 로 취급된다.
   - 별도의 래핑이나 추가 처리는 필요하지 않다.
2. `RunnableLambda()` 에 넣어 명시적으로 `Runnable` 타입으로 만든다.
   - Lambda 표현식으로 정의할 경우 `RunnableLambda(lambda 표현식)` 으로 정의해야 한다.
   - 보통 일반함수는 `RunnableLambda`를 사용할 필요 없다.
3. `@chain` decorator를 사용
   - 함수에 `@chain` decorator가 선언되면 그 함수는 `RunnableLambda` 타입이 된다.
   - 이 방식은 LCEL만으로 표현하기 어려운 실행 흐름을 직접 정의해야 할 때 주로 사용된다.
     - LCEL은 순차 실행구조를 따른다. 그래서  제어문을 이용해 그 흐름을 제어할 수가 없다. 
     - 단순한 파이프라인에서는 LCEL만으로도 충분하지만, 다음과 같은 경우에는 한계가 있다.
       - 특정 단계를 조건에 따라 실행하거나 생략해야 하는 경우
       - 동일한 단계를 반복적으로 실행해야 하는 경우
       - 여러 판단 로직에 따라 실행 경로가 달라지는 agent 구조
     - 이처럼 복잡한 업무 흐름을 가지는 agent는 단순한 순차 구조만으로는 원하는 응답 품질을 얻기 어렵다. 결국 실행 흐름 자체를 개발자가 직접 코드로 정의해야 하며, 이러한 경우에 `@chain`을 사용해 chain/agent 함수를 구현한다.
   - 이러한 복잡한 실행 흐름을 보다 구조적으로 정의하기 위해 LangChain에서 추가로 제공하는 것이 **LangGraph**이다.

In [29]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

def plus(n1, n2):
    return n1 + n2 

def wrap_plus(x):
    return plus(x[0], x[1])

# chain = RunnablePassthrough() | RunnableLambda(lambda x : plus(x[0], x[1]))
chain = RunnablePassthrough() | wrap_plus
chain.invoke([1, 2])

3

In [33]:
# recipe_chain 과 translate_chain을 이용해서 음식레시피를 특정 언어로 반환.
### recipe_chain의 결과와 translate_chain의 결과 둘을 모두 반환.
from langchain_core.runnables import chain

# RunnableLambda(multi_language_recipe_chain)

@chain
def multi_language_recipe_chain(input_data: dict) -> dict[str, str]:
    food: str = input_data['food']  # 음식 이름
    language: str = input_data['language'] # 레시피 언어
    is_korean: bool = input_data["is_korean"] # 한국어 레시피 필요 여부. 

    korean_recipe = recipe_chain.invoke({"food":food})

    result = translate_chain.invoke({"content": korean_recipe, "language": language})

    final_result = {"recipe": result}

    if is_korean: # 한국어 레시피도 요청
        final_result["korean_recipe"] = korean_recipe

    return final_result

type(multi_language_recipe_chain)

langchain_core.runnables.base.RunnableLambda

In [38]:
res = multi_language_recipe_chain.invoke(
    {"food":"돈까스", "language":"중국어", "is_korean": False}
)

In [39]:
res.keys()

dict_keys(['recipe'])

In [40]:
# print(res['korean_recipe'])
print(res['recipe'])

# 猪排饭团／炸猪排食谱

## 菜谱基本信息
- **难度**：中等
- **烹饪时间**：约30分钟
- **份量**：2人份

## 所需材料
- 猪里脊或猪里脊肉排 2片
- 盐 少许
- 黑胡椒 少许
- 面粉 1/2杯
- 鸡蛋 2个
- 面包糠 1～2杯
- 食用油 适量

### 可选配料
- 炸猪排酱
- 卷心菜丝
- 米饭
- 柠檬片

## 制作方法
1. **处理猪肉**  
   用刀背或肉锤轻轻拍打猪肉，使厚度均匀。不要拍得太厚，以免不易熟透。

2. **调味**  
   在处理好的猪肉正反面轻轻撒上盐和黑胡椒，做基础调味。

3. **准备裹粉材料**  
   将面粉、打散的鸡蛋液、面包糠分别放在不同的盘子里备用。

4. **裹粉**  
   先把猪肉均匀裹上一层面粉，再蘸上蛋液，最后充分按压裹上面包糠。

5. **预热油锅**  
   在平底锅中倒入足量食用油，用中火加热。  
   当撒入少许面包糠后会立刻浮起时，就表示可以开始炸了。

6. **油炸**  
   放入猪排，正反面炸至金黄。  
   每面约炸2～3分钟，直到整体呈金黄色并熟透。

7. **沥油**  
   将炸好的猪排放在厨房纸巾或网架上，沥去多余油分。

8. **切块**  
   稍微放凉后，切成适合入口的大小，摆盘。

9. **搭配上桌**  
   淋上炸猪排酱或单独装盘，搭配卷心菜丝和米饭一起享用。

## 小贴士
- 猪肉不要拍得过薄，否则油炸时容易破裂，适当厚度更好。
- 用手轻轻按压面包糠，可以让炸衣在油炸时更不容易脱落。
- 油温太低会吸油，太高则容易外面焦、里面不熟，保持中火最合适。
- 如果想更酥脆，可以再裹一层面包糠。


# Cache

- 응답 결과를 저장해서 같은 질문이 들어오면 LLM에 요청하지 않고 저장된 결과를 보여주도록 한다.
    - 처리속도와 비용을 절감할 수 있다.
    - 특히 chatbot같이 비슷한 질문을 하는 경우 유용하다.
- 저장 방식은 `메모리`, `sqlite` 등 다양한 방식을 지원한다.
  
    ```python
    set_llm_cache(Cache객체)
    ```

In [41]:
!uv pip install langchain-community

Resolved 45 packages in 1.39s
 Downloaded langchain-community
Prepared 8 packages in 1.24s
Installed 10 packages in 569ms
 + aiohappyeyeballs==2.6.2
 + aiohttp==3.14.1
 + aiosignal==1.4.0
 + frozenlist==1.8.0
 + httpx-sse==0.4.3
 + langchain-community==0.4.2
 + multidict==6.7.1
 + propcache==0.5.2
 + pydantic-settings==2.14.2
 + yarl==1.24.2


In [49]:
from langchain_core.globals import set_llm_cache
from langchain_community.cache import InMemoryCache, SQLiteCache

# Cache 설정은 한번만 하면된다.
# set_llm_cache(InMemoryCache())
set_llm_cache(SQLiteCache("cache.sqlite")) # cache를 저장할 파일 경로

In [57]:
res = multi_language_recipe_chain.invoke(
    {"food":"돈까스.", "language":"중국어", "is_korean": True}
)

In [56]:
print(res['korean_recipe'])

# 돈까스 레시피

## 요리 이름
- 돈까스

## 요리 기본정보
- **난이도:** 중
- **조리시간:** 약 30~40분
- **인분:** 2인분

## 요리에 필요한 재료
- 돼지고기 등심 또는 안심 2장
- 소금 약간
- 후추 약간
- 밀가루 1/2컵
- 달걀 2개
- 빵가루 2컵
- 식용유 넉넉히

### 선택 재료
- 양배추 채 썬 것
- 돈까스 소스
- 밥
- 우스터소스 또는 케첩

## 요리 방법
1. **고기 손질하기**  
   돼지고기는 칼등이나 고기망치로 가볍게 두드려 두께를 고르게 해줍니다.  
   너무 두꺼우면 속까지 익는 시간이 길어질 수 있으니 적당히 펴주세요.

2. **간하기**  
   손질한 고기에 소금과 후추를 앞뒤로 골고루 뿌려 밑간합니다.

3. **튀김옷 준비하기**  
   접시에 **밀가루**, 다른 접시에 **풀어둔 달걀**, 또 다른 접시에 **빵가루**를 각각 준비합니다.

4. **튀김옷 입히기**  
   밑간한 고기에  
   - 밀가루를 얇게 묻히고  
   - 달걀물에 적신 뒤  
   - 빵가루를 골고루 눌러 붙입니다.  
   빵가루가 잘 붙도록 손으로 살짝 눌러주세요.

5. **기름에 튀기기**  
   팬에 식용유를 넉넉히 붓고 중불로 달굽니다.  
   기름 온도가 올라가면 돈까스를 넣고 노릇노릇하게 튀깁니다.  
   한 면당 약 2~3분씩 튀긴 뒤 뒤집어 익혀주세요.

6. **기름 빼기**  
   다 익은 돈까스는 키친타월이나 철망 위에 올려 기름을 빼줍니다.

7. **먹기 좋게 썰기**  
   한 김 식힌 뒤 먹기 좋은 크기로 썰어 접시에 담습니다.  
   양배추 채와 밥, 돈까스 소스를 곁들이면 완성입니다.

## 팁
- **고기를 너무 두껍게 하지 않기**: 두께가 균일해야 속까지 골고루 익습니다.
- **빵가루를 꾹 눌러 붙이기**: 튀길 때 튀김옷이 잘 떨어지지 않습니다.
- **기름 온도 체크하기**: 너무 낮으면 기름을 많이 먹고, 너무 높으면 겉만 타기 쉽습

In [46]:
print(res['recipe'])

# 炸猪排食谱

## 基本料理信息
- **难度：** 中等
- **烹饪时间：** 约 30～40 分钟
- **份量：** 2人份

## 所需材料
### 主材料
- 猪里脊或猪腰肉 2片
- 少许盐
- 少许胡椒

### 裹粉材料
- 面粉 1/2 杯
- 鸡蛋 2个
- 面包糠 2杯

### 炸制用
- 适量食用油

### 配菜（可选）
- 切丝卷心菜
- 炸猪排酱
- 米饭

## 制作方法
1. **处理猪肉**  
   准备猪里脊或猪腰肉，将厚度整理均匀。  
   用肉锤或刀背轻轻敲打，使肉质更嫩。

2. **调味**  
   在处理好的猪肉两面均匀撒上盐和胡椒，静置约 5 分钟。

3. **准备裹衣材料**  
   在宽盘中分别放入面粉、打散的鸡蛋液和面包糠。

4. **裹粉**  
   先在猪肉表面薄薄地沾上一层面粉，再蘸上蛋液，最后均匀裹上面包糠。  
   用手轻轻按压，让面包糠更牢固地附着。

5. **加热油**  
   在平底锅中倒入足量食用油，加热至约 170～180℃。  
   放入少量面包糠后能立即浮起，温度就比较合适。

6. **油炸**  
   放入炸猪排，前后炸至金黄色。  
   不要一次放太多，期间适当翻面，使颜色均匀。

7. **沥油**  
   炸好的炸猪排放在厨房纸巾或网架上，沥去多余油分。

8. **完成**  
   切成方便入口的大小装盘，搭配卷心菜丝、炸猪排酱和米饭即可。

## 小贴士
- **猪肉不要切得太厚**，更容易熟透，口感也更好。
- **把面包糠轻轻按压牢固**，炸衣不容易脱落。
- **油温**太低会吸很多油，太高则容易外焦里生，请保持在 170～180℃。
- 如果想更酥脆，**油炸前可先放入冰箱静置 5～10 分钟**。


In [54]:
print(res['korean_recipe'])

# 돈까스 레시피

## 요리 기본정보
- **난이도:** 중
- **조리시간:** 약 30~40분
- **인분:** 2인분

## 요리에 필요한 재료
- 돼지고기 등심 2장(약 200~250g)
- 소금 약간
- 후추 약간
- 밀가루 1/2컵
- 달걀 2개
- 빵가루 1.5~2컵
- 식용유 넉넉히

### 선택 재료
- 양배추 채 썬 것
- 돈까스 소스
- 밥
- 레몬 1조각

## 요리 방법
1. **고기 손질하기**  
   돼지고기 등심은 키친타월로 핏물을 닦고, 칼등이나 고기망치로 두드려 두께를 고르게 만듭니다. 너무 두꺼우면 익는 시간이 길어지고 질길 수 있습니다.

2. **간하기**  
   손질한 고기 양면에 소금과 후추를 약간씩 뿌려 밑간합니다.

3. **튀김옷 준비하기**  
   접시 3개를 준비해 각각 밀가루, 풀어둔 달걀, 빵가루를 담습니다.

4. **옷 입히기**  
   밑간한 고기에 **밀가루 → 달걀물 → 빵가루** 순서로 골고루 입힙니다.  
   빵가루는 손으로 살짝 눌러 잘 붙게 해주세요.

5. **기름 예열하기**  
   팬에 식용유를 넉넉히 붓고 중불로 예열합니다.  
   빵가루를 조금 떨어뜨렸을 때 바로 올라오면 적당한 온도입니다.

6. **튀기기**  
   돈까스를 넣고 한 면당 2~3분씩 노릇하게 튀깁니다.  
   너무 센 불은 겉만 타고 속이 덜 익을 수 있으니 주의하세요.

7. **기름 빼기**  
   튀긴 돈까스는 체나 키친타월 위에 올려 기름을 빼줍니다.

8. **완성하기**  
   먹기 좋게 썰어 접시에 담고, 양배추 샐러드와 돈까스 소스를 곁들여 내면 완성입니다.

## 팁
- 고기를 **미리 냉장고에서 10분 정도 꺼내 두면** 튀길 때 온도 차가 줄어 더 고르게 익습니다.
- 빵가루는 **너무 세게 누르지 말고 살짝만** 눌러야 바삭한 식감이 살아납니다.
- 한 번에 너무 많이 넣으면 기름 온도가 떨어져 눅눅해질 수 있으니 **한두 장씩 튀기는 것**이 좋습니다.
- 더 바삭하게